# BERT requirement classification: improved experiments (Colab)

**Before you run:** Runtime -> Change runtime type -> **T4 GPU**.

What this notebook does (matches the improvement plan for the paper):
1. Loads PROMISE NFR (625 requirements, 15 projects).
2. Full fine-tuning (no frozen layers) of DistilBERT, BERT-base and RoBERTa-base, plus a TF-IDF+SVM baseline.
3. Seeds 42, 43, 44 under random 10-fold and project-held-out 5-fold, for Task A (FR vs NFR) and Task B (US/SE/O/PE).
4. Macro-F1 table (mean +- SD over seeds).
5. Calibration: ECE, cross-fitted temperature scaling, coverage-vs-precision curves.
6. Cluster bootstrap (resampling projects) for 95% confidence intervals of the F1 difference vs the SVM.

Results are saved after every model/seed, so if Colab disconnects, just re-run the cells and it resumes.
Expected time on a T4: roughly 1.5-3 hours for everything. To test quickly, set `SEEDS = [42]` in the config cell.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "scikit-learn", "scipy", "matplotlib", "pandas"], check=False)

import os, csv, time, random, urllib.request
import numpy as np, pandas as pd, torch
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", dev)
if dev == "cpu":
    print("WARNING: no GPU. Switch the runtime to T4 GPU or this will take far too long.")

In [ ]:
# Where results are stored. Drive keeps them safe if Colab disconnects.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/bert_paper_results"
else:
    OUT = "results"
os.makedirs(OUT, exist_ok=True)
print("Saving to:", OUT)

In [ ]:
# Load PROMISE NFR from Zenodo (record 268542). If the download fails, upload nfr.arff into ./data/ by hand.
os.makedirs("data", exist_ok=True)
path = "data/nfr.arff"
URLS = ["https://zenodo.org/records/268542/files/nfr.arff",
        "https://zenodo.org/record/268542/files/nfr.arff"]
if not os.path.exists(path):
    for u in URLS:
        try:
            urllib.request.urlretrieve(u, path); break
        except Exception as e:
            print("Download failed for", u, "->", e)

rows, in_data = [], False
with open(path, encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()
        if line.lower().startswith("@data"):
            in_data = True; continue
        if in_data and line and not line.startswith("%"):
            rows.append(next(csv.reader([line], quotechar="'", escapechar="\\")))

df = pd.DataFrame(rows, columns=["project", "text", "cls"])
df["project"] = df.project.astype(int)
df["cls"] = df.cls.str.strip()
assert len(df) == 625, f"Expected 625 requirements, got {len(df)}. Check the ARFF parsing."
df["fr"] = np.where(df.cls == "F", "FR", "NFR")
print(df.cls.value_counts().to_string())
print("Projects:", df.project.nunique())

TASKS = {
    "A": (df.copy(), "fr"),
    "B": (df[df.cls.isin(["US", "SE", "O", "PE"])].reset_index(drop=True), "cls"),
}
print({k: len(v[0]) for k, v in TASKS.items()})

In [ ]:
# Configuration
MODELS = {
    "distilbert": ("distilbert-base-uncased", 5e-5),
    "bert":       ("bert-base-uncased", 3e-5),
    "roberta":    ("roberta-base", 3e-5),
}
SEEDS = [42, 43, 44]
EPOCHS, BATCH, MAX_LEN = 5, 16, 64   # same epochs and max length as the paper; batch 16 on GPU

def get_folds(d, label, protocol, seed):
    y = d[label].values
    if protocol == "random":
        sp = StratifiedKFold(10, shuffle=True, random_state=seed).split(d, y)
    else:
        sp = StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(d, y, groups=d.project.values)
    return list(sp)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def finetune_predict(model_name, lr, tr_text, tr_y, te_text, n_cls, seed):
    # Full fine-tuning (nothing frozen), class-weighted loss as in Eq. (1) of the paper.
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=n_cls).to(dev)
    enc = lambda t: tok(list(t), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
    Xtr, Xte = enc(tr_text), enc(te_text)
    ytr = torch.tensor(tr_y)
    counts = np.bincount(tr_y, minlength=n_cls).clip(min=1)
    w = torch.tensor(len(tr_y) / (n_cls * counts), dtype=torch.float, device=dev)
    lossf = torch.nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = EPOCHS * int(np.ceil(len(tr_y) / BATCH))
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    model.train()
    for ep in range(EPOCHS):
        perm = torch.randperm(len(tr_y))
        for i in range(0, len(perm), BATCH):
            idx = perm[i:i + BATCH]
            out = model(input_ids=Xtr["input_ids"][idx].to(dev),
                        attention_mask=Xtr["attention_mask"][idx].to(dev))
            loss = lossf(out.logits, ytr[idx].to(dev))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); opt.zero_grad()
    model.eval(); logits = []
    with torch.no_grad():
        for i in range(0, len(te_text), 64):
            o = model(input_ids=Xte["input_ids"][i:i + 64].to(dev),
                      attention_mask=Xte["attention_mask"][i:i + 64].to(dev))
            logits.append(o.logits.float().cpu())
    del model, opt
    if dev == "cuda": torch.cuda.empty_cache()
    return torch.cat(logits).numpy()

def svm_predict(tr_text, tr_y, te_text):
    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english")
    clf = LinearSVC(C=1, class_weight="balanced").fit(vec.fit_transform(tr_text), tr_y)
    return clf.predict(vec.transform(te_text))

In [ ]:
# Main experiment loop (resumable: finished runs are skipped)
def run_all():
    for task, (d, label) in TASKS.items():
        classes = sorted(d[label].unique())
        cmap = {c: i for i, c in enumerate(classes)}
        y = d[label].map(cmap).values
        for protocol in ["random", "project"]:
            for seed in SEEDS:
                folds = get_folds(d, label, protocol, seed)
                fold_id = np.zeros(len(d), int)
                for k, (_, te) in enumerate(folds): fold_id[te] = k
                for name in ["svm"] + list(MODELS):
                    f = f"{OUT}/{task}_{protocol}_{name}_{seed}.npz"
                    if os.path.exists(f): continue
                    t0 = time.time()
                    logits = np.zeros((len(d), len(classes)), np.float32)
                    pred = np.zeros(len(d), int)
                    for k, (tr, te) in enumerate(folds):
                        if name == "svm":
                            pred[te] = svm_predict(d.text.values[tr], y[tr], d.text.values[te])
                        else:
                            mn, lr = MODELS[name]
                            lg = finetune_predict(mn, lr, d.text.values[tr], y[tr], d.text.values[te], len(classes), seed)
                            logits[te] = lg; pred[te] = lg.argmax(1)
                    np.savez(f, pred=pred, logits=logits, y=y, fold=fold_id, project=d.project.values)
                    mf1 = f1_score(y, pred, average="macro")
                    print(f"task {task} | {protocol:7s} | {name:10s} | seed {seed} | macro-F1 {mf1:.3f} | {time.time()-t0:.0f}s")

run_all()

In [ ]:
# Table 1: macro-F1, mean +- SD over seeds (use this to replace Table III)
recs = []
for task in TASKS:
    for protocol in ["random", "project"]:
        for name in ["svm"] + list(MODELS):
            for seed in SEEDS:
                f = f"{OUT}/{task}_{protocol}_{name}_{seed}.npz"
                if os.path.exists(f):
                    z = np.load(f)
                    recs.append(dict(task=task, protocol=protocol, model=name, seed=seed,
                                     macro_f1=f1_score(z["y"], z["pred"], average="macro")))
res = pd.DataFrame(recs)
res.to_csv(f"{OUT}/per_seed_results.csv", index=False)
summary = (res.groupby(["model", "task", "protocol"]).macro_f1.agg(["mean", "std"]).round(3)
              .assign(txt=lambda t: t["mean"].astype(str) + " +- " + t["std"].astype(str))["txt"]
              .unstack(["task", "protocol"]))
summary.to_csv(f"{OUT}/table_macro_f1.csv")
summary

In [ ]:
# Calibration: ECE, cross-fitted temperature scaling, coverage vs precision
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

def softmax(z, T=1.0):
    z = z / T; z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

def ece(p, y, bins=15):
    conf, acc = p.max(1), (p.argmax(1) == y)
    edges, e = np.linspace(0, 1, bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e

def fit_T(z, y):
    nll = lambda T: -np.log(softmax(z, T)[np.arange(len(y)), y] + 1e-12).mean()
    return minimize_scalar(nll, bounds=(0.05, 10), method="bounded").x

def crossfit_probs(z, y, fold):
    # Temperature for each fold is fitted only on the OTHER folds' out-of-fold logits (no leakage).
    p = np.zeros_like(z)
    for k in np.unique(fold):
        m = fold == k
        p[m] = softmax(z[m], fit_T(z[~m], y[~m]))
    return p

taus = np.linspace(0.5, 0.99, 50)
def cov_prec(p, y):
    conf, ok = p.max(1), p.argmax(1) == y
    cov = np.array([(conf >= t).mean() for t in taus])
    prec = np.array([ok[conf >= t].mean() if (conf >= t).any() else np.nan for t in taus])
    return cov, prec

rows, curves = [], {}
for task in TASKS:
    for protocol in ["random", "project"]:
        for name in MODELS:
            cr, cc, pr, pc = [], [], [], []
            for seed in SEEDS:
                f = f"{OUT}/{task}_{protocol}_{name}_{seed}.npz"
                if not os.path.exists(f): continue
                z = np.load(f); y, lg, fold = z["y"], z["logits"], z["fold"]
                p_raw = softmax(lg); p_cal = crossfit_probs(lg, y, fold)
                rows.append(dict(task=task, protocol=protocol, model=name, seed=seed,
                                 ECE_raw=ece(p_raw, y), ECE_calibrated=ece(p_cal, y)))
                a, b = cov_prec(p_raw, y); c, d_ = cov_prec(p_cal, y)
                cr.append(a); pr.append(b); cc.append(c); pc.append(d_)
            if cr:
                curves[(task, protocol, name)] = (np.mean(cr, 0), np.nanmean(pr, 0), np.mean(cc, 0), np.nanmean(pc, 0))

cal = pd.DataFrame(rows)
cal_summary = cal.groupby(["task", "protocol", "model"])[["ECE_raw", "ECE_calibrated"]].mean().round(3)
cal_summary.to_csv(f"{OUT}/table_calibration.csv")
print(cal_summary)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
colors = {"distilbert": "tab:blue", "bert": "tab:orange", "roberta": "tab:green"}
for ax, (task, protocol) in zip(axes.ravel(), [(t, p) for t in TASKS for p in ["random", "project"]]):
    for name in MODELS:
        if (task, protocol, name) in curves:
            cr_, pr_, cc_, pc_ = curves[(task, protocol, name)]
            ax.plot(cr_, pr_, "--", color=colors[name], alpha=0.6)
            ax.plot(cc_, pc_, "-", color=colors[name], label=name)
    ax.axhline(0.95, color="grey", lw=0.8, ls=":")
    ax.set_title(f"Task {task}, {protocol} (dashed = raw, solid = temp-scaled)", fontsize=9)
    ax.set_xlabel("Coverage (share auto-labelled)"); ax.set_ylabel("Precision of auto-labels")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/coverage_precision.png", dpi=200); plt.show()

In [ ]:
# Cluster bootstrap by project: 95% CI of macro-F1 difference (model minus SVM)
def macro(y, p, labels): return f1_score(y, p, average="macro", labels=labels, zero_division=0)

def cluster_boot(task, protocol, seed, name, n=1000):
    A = np.load(f"{OUT}/{task}_{protocol}_{name}_{seed}.npz")
    B = np.load(f"{OUT}/{task}_{protocol}_svm_{seed}.npz")
    y, proj, pa, pb = A["y"], A["project"], A["pred"], B["pred"]
    labels, projs = np.unique(y), np.unique(proj)
    by = {p: np.where(proj == p)[0] for p in projs}
    rng = np.random.default_rng(0); diffs = []
    for _ in range(n):
        ii = np.concatenate([by[p] for p in rng.choice(projs, len(projs), replace=True)])
        diffs.append(macro(y[ii], pa[ii], labels) - macro(y[ii], pb[ii], labels))
    obs = macro(y, pa, labels) - macro(y, pb, labels)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return obs, lo, hi

brow = []
for task in TASKS:
    for protocol in ["random", "project"]:
        for name in MODELS:
            for seed in SEEDS:
                if os.path.exists(f"{OUT}/{task}_{protocol}_{name}_{seed}.npz"):
                    o, lo, hi = cluster_boot(task, protocol, seed, name)
                    brow.append(dict(task=task, protocol=protocol, model=name, seed=seed,
                                     diff_vs_svm=round(o, 3), ci_low=round(lo, 3), ci_high=round(hi, 3),
                                     excludes_zero=(lo > 0) or (hi < 0)))
boot = pd.DataFrame(brow)
boot.to_csv(f"{OUT}/table_cluster_bootstrap.csv", index=False)
boot

In [ ]:
# Download everything (tables, figure, raw predictions)
import shutil
shutil.make_archive("bert_paper_results", "zip", OUT)
try:
    from google.colab import files
    files.download("bert_paper_results.zip")
except Exception as e:
    print("Zip saved as bert_paper_results.zip;", e)

## How to use these results in the paper
- **Table III**: replace with `table_macro_f1.csv` (full fine-tuning, 3 seeds everywhere). Delete the sentences about frozen layers and the one-seed caveat, and say the models were fully fine-tuned.
- **Table IV (McNemar)**: keep it, and add `table_cluster_bootstrap.csv` as the stronger test. Say that the intervals resample whole projects. If an interval includes 0, do not claim a difference.
- **Calibration**: add `table_calibration.csv` and `coverage_precision.png`. Report ECE before and after temperature scaling, and which tau reaches 0.95 precision and at what coverage.
- **Write whatever comes out.** Numbers will differ from the current draft. Update the abstract, Section IV and the conclusion to match, and do not keep sentences that the new results contradict.
- Upload this notebook and the CSVs to GitHub, archive it on Zenodo, and cite the DOI in the paper.